In [ ]:
%sql
/* SQL script to calculate and process applied quantity (apl_qty) in inventory transactions */

/* Create table purgo_playground.f_inv_movmnt_apl_qty if not exists */
CREATE TABLE IF NOT EXISTS purgo_playground.f_inv_movmnt_apl_qty (
  txn_id STRING NOT NULL COMMENT "Unique transaction identifier",
  ref_txn_qty DECIMAL(3,1) NOT NULL COMMENT "Reference transaction quantity, can be positive or negative",
  cumulative_txn_qty DECIMAL(4,1) NOT NULL COMMENT "Cumulative transaction quantity until the current date",
  cumulative_ref_ord_sched_qty DECIMAL(4,1) NOT NULL COMMENT "Cumulative reference order scheduled quantity until the current date",
  ref_ord_sched_qty DECIMAL(3,1) NOT NULL COMMENT "Reference order scheduled quantity",
  prior_cumulative_txn_qty DECIMAL(3,1) NOT NULL COMMENT "Prior cumulative transaction quantity before the current date",
  prior_cumulative_ref_ord_sched_qty DECIMAL(3,1) NOT NULL COMMENT "Prior cumulative reference order quantity before the current date",
  apl_qty DECIMAL(5,1) COMMENT "Calculated applied quantity based on the transaction logic"
) COMMENT "Table storing inventory transaction data with applied quantities";

/* Data Transformation CTE for calculating applied quantity (apl_qty) */
WITH CTE_Calculate_Applied_Qty AS (
  SELECT 
    txn_id,
    ref_txn_qty,
    cumulative_txn_qty,
    cumulative_ref_ord_sched_qty,
    ref_ord_sched_qty,
    prior_cumulative_txn_qty,
    prior_cumulative_ref_ord_sched_qty,
    CASE
      WHEN ref_txn_qty > 0 AND cumulative_txn_qty >= cumulative_ref_ord_sched_qty THEN 
        CASE
          WHEN prior_cumulative_ref_ord_sched_qty < prior_cumulative_txn_qty THEN 
            ref_ord_sched_qty - (prior_cumulative_txn_qty - prior_cumulative_ref_ord_sched_qty)
          ELSE ref_ord_sched_qty
        END
      WHEN ref_txn_qty > 0 AND cumulative_ref_ord_sched_qty >= cumulative_txn_qty THEN 
        CASE
          WHEN prior_cumulative_ref_ord_sched_qty > prior_cumulative_txn_qty THEN 
            ref_txn_qty - (prior_cumulative_ref_ord_sched_qty - prior_cumulative_txn_qty)
          ELSE ref_txn_qty
        END
      WHEN ref_txn_qty < 0 AND cumulative_txn_qty != 0 AND cumulative_ref_ord_sched_qty > 0 THEN
        ref_txn_qty
      ELSE NULL
    END AS apl_qty
  FROM purgo_playground.f_inv_movmnt_apl_qty
)

/* Insert calculated applied quantity data into target table */
INSERT INTO purgo_playground.f_inv_movmnt_apl_qty
SELECT 
  txn_id,
  ref_txn_qty,
  cumulative_txn_qty,
  cumulative_ref_ord_sched_qty,
  ref_ord_sched_qty,
  prior_cumulative_txn_qty,
  prior_cumulative_ref_ord_sched_qty,
  apl_qty
FROM CTE_Calculate_Applied_Qty;

/* End of SQL script */
